## EV Driver Behavior at EVCS
* Author: Callie Clark
* Output: proportion of if a user stays at car, visits POI or leaves their car and does not visit observed POI

In [ ]:
exec(open('../EV00_settings.py').read())

In [ ]:
!pip install pydeck -q -q
!pip install seaborn

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick

In [ ]:


def bootstrap_category_ci(df, n_bootstrap=1000):
    categories = ['Beauty', 'Entertainment', 'Errands', 'Fitness', 
                  'Grocery', 'Pharmacies', 'Restaurants', 'Retail']
    
    # Pre-extract the relevant columns as numpy arrays for speed
    category_data = df[categories].values
    n_samples = len(df)
    
    # Pre-allocate results array
    results = np.zeros((n_bootstrap, len(categories)))
    
    for i in range(n_bootstrap):
        # Generate bootstrap indices
        bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # Calculate means for this bootstrap sample
        bootstrap_sample = category_data[bootstrap_indices]
        results[i] = bootstrap_sample.mean(axis=0)
    print(results)
    # Compute statistics
    mean_values = results.mean(axis=0)
    lower_ci = np.percentile(results, 2.5, axis=0)
    upper_ci = np.percentile(results, 97.5, axis=0)
    
    # Convert back to pandas Series with proper index
    mean_values = pd.Series(mean_values, index=categories)
    lower_ci = pd.Series(lower_ci, index=categories)
    upper_ci = pd.Series(upper_ci, index=categories)
    
    return mean_values, lower_ci, upper_ci


## Fill in missing values 
* Run EV_99 first

In [ ]:
## number check 
counter_num=0
counter_den=0
val_ls=[]
for current_city in [current_city]:# in ['Denver', 'SF', 'Seattle', 'Boston']: 
    print (current_city)

    df_grey=pd.read_csv(f'{current_session_path}evcs_session_stops/unobserved_points/unobserved_points_dist_{current_city}.csv',index_col=0)
    df_grey['session_id']=df_grey['cuebiq_id'].astype(str)+'_'+df_grey['date'].astype(str)


    evcs_visits=pd.DataFrame()      
    # convert inputs to datetime
    start_dt = dt.strptime(start_date, '%Y%m%d')
    end_dt = dt.strptime(end_date, '%Y%m%d')


    current_date = start_dt
    while current_date <= end_dt:
        date_str = str(current_date.strftime('%Y%m%d'))
        #print(f"-------------------------------------------------\nProcessing date: {date_str}")


        evcs_stop_df=pd.read_csv(f'{current_session_path}evcs_session_stops/model/driver_EVCS_behavior_{current_city}_{date_str}.csv',index_col=0)
        #evcs_visits=evcs_stop_df.dropna(subset='placekey')
        evcs_visits=pd.concat([evcs_visits,evcs_stop_df])



        current_date = current_date + timedelta(days = 1)
    print(len(evcs_visits))
    evcs_visits=evcs_visits[~(evcs_visits.classification_type=='RECURRING_AREA')].copy()
    evcs_visits.category.value_counts()


    # --- 1. Filter to "no stop" ---
    no_stop_visits = evcs_visits.loc[
        evcs_visits["category"] == "no stop"
    ].copy()

    # --- 2. Merge EVCS reference coordinates ---
    print(len(no_stop_visits))
    no_stop_mapped = no_stop_visits.merge(
        df_grey[['session_id','evcs_id','stop_zoned_datetime','evcs_dist_75', 'evcs_dist_mean',
           'evcs_dist_max', 'num_points']],
        on=['session_id','evcs_id','stop_zoned_datetime'],
        how="inner")



    no_stop_mapped['evcs_dist']=no_stop_mapped['evcs_dist_75']
    no_stop_mapped['category'] = np.where(
        no_stop_mapped['evcs_dist_75'] <= 10,
        'evcs',
        'no poi')
    print(len(no_stop_mapped[no_stop_mapped['category']=='evcs'])/len(no_stop_mapped))
    print(no_stop_mapped[no_stop_mapped['category']=='no poi'].evcs_dist_75.mean())
    counter_num+=len(no_stop_mapped[no_stop_mapped['category']=='evcs'])
    counter_den+=len(no_stop_mapped)
    val_ls+=list(no_stop_mapped[no_stop_mapped['category']=='no poi'].evcs_dist_75)
    

    

In [ ]:

evcs_visits=pd.DataFrame()      
# convert inputs to datetime
start_dt = dt.strptime(start_date, '%Y%m%d')
end_dt = dt.strptime(end_date, '%Y%m%d')


current_date = start_dt
while current_date <= end_dt:
    date_str = str(current_date.strftime('%Y%m%d'))
    #print(f"-------------------------------------------------\nProcessing date: {date_str}")
    

    evcs_stop_df=pd.read_csv(f'{current_session_path}evcs_session_stops/model/driver_EVCS_behavior_{current_city}_{date_str}.csv',index_col=0)
    #evcs_visits=evcs_stop_df.dropna(subset='placekey')
    evcs_visits=pd.concat([evcs_visits,evcs_stop_df])

   
    
    current_date = current_date + timedelta(days = 1)
print(len(evcs_visits))
evcs_visits=evcs_visits[~(evcs_visits.classification_type=='RECURRING_AREA')].copy()
evcs_visits.category.value_counts()




In [ ]:


# --- 1. Filter to "no stop" ---
no_stop_visits = evcs_visits.loc[
    evcs_visits["category"] == "no stop"
].copy()

# --- 2. Merge EVCS reference coordinates ---
print(len(no_stop_visits))
no_stop_mapped = no_stop_visits.merge(
    df_grey[['session_id','evcs_id','stop_zoned_datetime','evcs_dist_75', 'evcs_dist_mean',
       'evcs_dist_max', 'num_points']],
    on=['session_id','evcs_id','stop_zoned_datetime'],
    how="inner")
print(len(no_stop_mapped))


# # Optional: inspect results
# no_stop_visits[[
#     "evcs_id", "lat", "lng", "Latitude", "Longitude", "dist_to_evcs_m"
# ]].head()
no_stop_mapped.head()

In [ ]:

no_stop_mapped['evcs_dist']=no_stop_mapped['evcs_dist_75']
no_stop_mapped['category'] = np.where(
    no_stop_mapped['evcs_dist_75'] <= 10,
    'evcs',
    'no poi')
no_stop_mapped['category'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

cols = [ 'evcs_dist_max','evcs_dist_75', 'evcs_dist_mean',]

plt.figure(figsize=(8, 5))

for col in cols:
    plt.hist(
        no_stop_mapped[no_stop_mapped.evcs_dist_max<500][col].dropna(),
        bins=50,
        alpha=0.5,
        label=col
    )
plt.axvline(
    x=10,
    linestyle='--',
    linewidth=2,
    label='10'
)
plt.xlabel('Distance')
plt.ylabel('Count')
plt.legend()
plt.title(f'{current_city}: Overlapping Histogram of EVCS Distances')
plt.tight_layout()
plt.show()


In [ ]:
evcs_visits_mapped=pd.concat([evcs_visits.loc[
    evcs_visits["category"] != "no stop"],no_stop_mapped[['session_id', 'cuebiq_id', 'stop_zoned_datetime', 'dwell_time_minutes',
       'lat', 'lng', 'evcs_dist', 'T_since_charge', 'classification_type',
       'evcs_id', 'placekey', 'category', 'name', 'geometry', 'GEOID']]]).reset_index(drop=True)
evcs_visits_mapped=evcs_visits_mapped.drop_duplicates(subset=['cuebiq_id', 'evcs_id','stop_zoned_datetime'])
evcs_visits_mapped.to_csv(f'{current_session_path}evcs_session_stops/model/driver_EVCS_behavior_{current_city}_cleaned_2.csv')


## Run mapped data

In [ ]:
#evcs_visits_mapped=pd.read_csv(f'{current_session_path}evcs_session_stops/model/driver_EVCS_behavior_{current_city}_cleaned_2.csv')


In [ ]:


# Label mapping (for the legend)
label_map = {
    'no stop': 'Unobserved',
    'evcs': 'Stay at EVCS',
    'no poi': 'Leave Car, No POI Visit',
    'Poi visit': 'POI Visit'
}


color_map = {
    'no stop': '#c0c0c0',   # neutral gray
    'evcs': '#4daf4a',      # natural green
    'no poi': '#a6bddb',    # pale desaturated blue
    'Poi visit': '#0571b0'  # rich deep blue
}


def categorize(val):
    if val in ['no poi', 'evcs', 'no stop']:
        return val
    else:
        return 'Poi visit'

def prepare_counts(df):
    s = df['category'].apply(categorize)
    return s.value_counts(normalize=True)

# Prepare both datasets
#counts_city = prepare_counts(evcs_visits)
counts_city = prepare_counts(evcs_visits_mapped)

# Desired order
#all_cats = ['no stop', 'evcs',  'no poi','Poi visit',]
all_cats = [ 'evcs',  'no poi','Poi visit',]
counts_city = counts_city.reindex(all_cats, fill_value=0)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(6, 6), sharey=True)

x = range(len(all_cats))


colors = [color_map[c] for c in all_cats]
ax.bar(x, counts_city.values, color=colors, width=0.7)

ax.set_title(current_city, fontsize=18, y=1.06)
ax.set_yticks([])
#ax.set_ylim(0, 1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
for i, v in enumerate(counts_city.values):
    ax.text(i, v + 0.02, f"{v:.2%}", 
            ha='center', va='bottom', fontsize=14)


ax.set_ylabel("Percent of Visits", fontsize=16)
ax.tick_params(axis='x', labelsize=14)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='y', which='both', length=0)
#ax.tick_params(axis='y', which='both', length=0)

# Legend patches in correct order
handles = [mpatches.Patch(color=color_map[c], label=label_map[c]) for c in all_cats]

# Add suptitle higher
plt.suptitle(f"{current_city}: EV Driver Behavior During Charging Session", fontsize=22, y=1.08)

# Add legend directly under the suptitle, spanning width
fig.legend(handles=handles,
           loc='upper center',
           bbox_to_anchor=(0.5, 1.02),
           ncol=4,
           frameon=False,
           fontsize=14)

plt.subplots_adjust(top=0.82, bottom=0.12)  # leave space for top legend and suptitle
#plt.savefig(os.path.join(f"../../figures/{current_city}_evcs_session_behavior.png"), dpi=300)

plt.show()
